
# Notebook 03 — Baseline Supply Plan

## Objective
Develop a rule-based, time-phased supply plan for
a hypothetical hardware supply network using
the validated inputs from Notebook 02.

## Planning Scope
- 12-week planning horizon
- 1 finished hardware product
- 3 component groups
- 2 suppliers
- 1 manufacturing plant
- 2 distribution centers
- East and West customer markets

## Baseline Strategy
Prioritize lower-cost procurement, schedule
production to meet future requirements, and
use predefined transportation routes.

Track supplier lead times, component availability,
production, shipments, distribution inventory,
customer fulfillment, and operating costs.

## Key Outputs
1. Weekly supplier purchase orders
2. Weekly production plan
3. Transportation shipment schedule
4. Inventory balances and shortages
5. Total baseline supply network cost

## Methodology
This notebook uses rule-based planning rather
than mathematical optimization.

The resulting baseline will be compared against
the optimized plan developed in Notebook 04.

All supply-network parameters are hypothetical.

In [1]:
# STEP 2: Load validated supply network inputs

import pandas as pd
import numpy as np
from pathlib import Path

# Locate Project 2 repository
current_path = Path.cwd().resolve()

repo_root = next(
    (
        path for path in
        [current_path, *current_path.parents]
        if (path / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# Input dataset paths
input_files = {
    "bom": "data/assumptions/bom.csv",
    "suppliers": "data/assumptions/suppliers.csv",
    "plants": "data/assumptions/plants.csv",
    "dcs": "data/assumptions/distribution_centers.csv",
    "transport": "data/assumptions/transport_lanes.csv",
    "inventory": "data/assumptions/initial_inventory.csv",
    "allocation": "data/assumptions/market_allocation.csv",
    "demand": "data/processed/demand_plan_12_weeks.csv",
    "regional": "data/processed/regional_demand_12_weeks.csv"
}

# Load datasets
inputs = {}

for name, relative_path in input_files.items():

    file_path = repo_root / relative_path

    assert file_path.is_file(), (
        f"Missing dataset: {file_path}"
    )

    inputs[name] = pd.read_csv(file_path)

# Confirm data integrity
assert len(inputs) == 9
assert len(inputs["demand"]) == 12
assert len(inputs["regional"]) == 24

assert np.isclose(
    inputs["demand"]["Demand_Units"].sum(),
    inputs["regional"]["Demand_Units"].sum()
)

print("Notebook 03 initialized successfully!")
print("Datasets loaded:", len(inputs))
print("Planning weeks:", len(inputs["demand"]))
print(
    "Total demand:",
    inputs["regional"]["Demand_Units"].sum()
)

print("\nDATASET INVENTORY:")

dataset_summary = pd.DataFrame([
    {"Dataset": name, "Records": len(df)}
    for name, df in inputs.items()
])

display(dataset_summary)


Notebook 03 initialized successfully!
Datasets loaded: 9
Planning weeks: 12
Total demand: 6180

DATASET INVENTORY:


,Dataset,Records
0,bom,3
1,suppliers,6
2,plants,1
3,dcs,2
4,transport,4
5,inventory,6
6,allocation,2
7,demand,12
8,regional,24


In [2]:
# STEP 3A: Define baseline production policy

demand_data = inputs["demand"]
plant_data = inputs["plants"]

# Get weekly demand
weekly_demand = (
    demand_data.groupby("Week")["Demand_Units"]
    .sum()
    .sort_index()
)

# Confirm baseline demand assumptions
assert len(weekly_demand) == 12
assert weekly_demand.nunique() == 1

# Baseline production policy
production_plan = pd.DataFrame({
    "Week": weekly_demand.index,
    "Demand_Units": weekly_demand.values
})

# Produce in Weeks 1-11; Week 12 demand
# is served by earlier production and inventory
production_plan["Production_Units"] = np.where(
    production_plan["Week"] <= 11,
    production_plan["Demand_Units"],
    0
).astype(int)

production_plan["Plant_ID"] = "PLANT_01"
production_plan["Planning_Method"] = "Rule_Based_Baseline"

# Validate production capacity
plant_capacity = int(
    plant_data.loc[
        plant_data["Plant_ID"] == "PLANT_01",
        "Weekly_Capacity_Units"
    ].iloc[0]
)

assert (
    production_plan["Production_Units"]
    <= plant_capacity
).all()

print("Baseline production schedule created!")
print(
    "Total planned production:",
    production_plan["Production_Units"].sum()
)
print(
    "Maximum weekly production:",
    production_plan["Production_Units"].max()
)
print("Plant weekly capacity:", plant_capacity)

display(production_plan)


Baseline production schedule created!
Total planned production: 5665
Maximum weekly production: 515
Plant weekly capacity: 580


,Week,Demand_Units,Production_Units,Plant_ID,Planning_Method
0,1,515.0,515,PLANT_01,Rule_Based_Baseline
1,2,515.0,515,PLANT_01,Rule_Based_Baseline
2,3,515.0,515,PLANT_01,Rule_Based_Baseline
3,4,515.0,515,PLANT_01,Rule_Based_Baseline
4,5,515.0,515,PLANT_01,Rule_Based_Baseline
5,6,515.0,515,PLANT_01,Rule_Based_Baseline
6,7,515.0,515,PLANT_01,Rule_Based_Baseline
7,8,515.0,515,PLANT_01,Rule_Based_Baseline
8,9,515.0,515,PLANT_01,Rule_Based_Baseline
9,10,515.0,515,PLANT_01,Rule_Based_Baseline


In [3]:
# STEP 4: Build baseline purchase order schedule

supplier_master = inputs["suppliers"]
bom_master = inputs["bom"]

orders = []

for component_id in bom_master["Component_ID"]:

    # Supplier A: 450 units in Weeks 1-9
    for week in range(1, 10):
        orders.append({
            "Week": week,
            "Supplier_ID": "SUP_A",
            "Component_ID": component_id,
            "Order_Units": 450
        })

    # Supplier B: 65 units in Weeks 1-7
    for week in range(1, 8):
        orders.append({
            "Week": week,
            "Supplier_ID": "SUP_B",
            "Component_ID": component_id,
            "Order_Units": 65
        })

    # Final supplemental purchase in Week 8
    orders.append({
        "Week": 8,
        "Supplier_ID": "SUP_B",
        "Component_ID": component_id,
        "Order_Units": 60
    })

purchase_orders = pd.DataFrame(orders)

# Add actual supplier cost, capacity and lead times
purchase_orders = purchase_orders.merge(
    supplier_master[[
        "Supplier_ID",
        "Component_ID",
        "Unit_Cost_USD",
        "Weekly_Capacity_Units",
        "Lead_Time_Weeks"
    ]],
    on=["Supplier_ID", "Component_ID"],
    how="left",
    validate="many_to_one"
)

# Calculate receipt weeks and purchase cost
purchase_orders["Arrival_Week"] = (
    purchase_orders["Week"]
    + purchase_orders["Lead_Time_Weeks"]
)

purchase_orders["Purchase_Cost_USD"] = (
    purchase_orders["Order_Units"]
    * purchase_orders["Unit_Cost_USD"]
)

# Basic validations
assert purchase_orders.notna().all().all()
assert (
    purchase_orders["Order_Units"]
    <= purchase_orders["Weekly_Capacity_Units"]
).all()

assert purchase_orders["Arrival_Week"].le(12).all()

# Check receipts plus beginning inventory
initial_components = inputs["inventory"].query(
    "Item_Type == 'Component'"
).set_index("Item_ID")["Beginning_Inventory_Units"]

required_components = (
    bom_master.set_index("Component_ID")[
        "Units_Per_Finished_Unit"
    ] * production_plan["Production_Units"].sum()
)

ordered_components = purchase_orders.groupby(
    "Component_ID"
)["Order_Units"].sum()

component_balance = pd.DataFrame({
    "Beginning_Inventory": initial_components,
    "Ordered_Units": ordered_components,
    "Required_Units": required_components
})

component_balance["Projected_End_Units"] = (
    component_balance["Beginning_Inventory"]
    + component_balance["Ordered_Units"]
    - component_balance["Required_Units"]
)

assert component_balance["Projected_End_Units"].eq(0).all()

print("Baseline purchase schedule created!")
print("Purchase order lines:", len(purchase_orders))
print(
    "Total procurement cost: $",
    round(purchase_orders["Purchase_Cost_USD"].sum(), 2)
)

print("\nCOMPONENT BALANCE:")
display(component_balance)

print("\nPURCHASE ORDER PREVIEW:")
display(purchase_orders.head(10))


Baseline purchase schedule created!
Purchase order lines: 51
Total procurement cost: $ 254235

COMPONENT BALANCE:


,Beginning_Inventory,Ordered_Units,Required_Units,Projected_End_Units
BATT_001,1100,4565,5665,0
ELEC_001,1100,4565,5665,0
ENCL_001,1100,4565,5665,0



PURCHASE ORDER PREVIEW:


,Week,Supplier_ID,Component_ID,Order_Units,Unit_Cost_USD,Weekly_Capacity_Units,Lead_Time_Weeks,Arrival_Week,Purchase_Cost_USD
0,1,SUP_A,ELEC_001,450,32,450,2,3,14400
1,2,SUP_A,ELEC_001,450,32,450,2,4,14400
2,3,SUP_A,ELEC_001,450,32,450,2,5,14400
3,4,SUP_A,ELEC_001,450,32,450,2,6,14400
4,5,SUP_A,ELEC_001,450,32,450,2,7,14400
5,6,SUP_A,ELEC_001,450,32,450,2,8,14400
6,7,SUP_A,ELEC_001,450,32,450,2,9,14400
7,8,SUP_A,ELEC_001,450,32,450,2,10,14400
8,9,SUP_A,ELEC_001,450,32,450,2,11,14400
9,1,SUP_B,ELEC_001,65,40,250,1,2,2600


In [5]:

# STEP 5A: Generate baseline transportation schedule

# Select the baseline transportation modes
transport_master = inputs["transport"]
regional = inputs["regional"]

selected_lanes = transport_master.loc[
    (
        (transport_master["Destination_DC_ID"] == "DC_EAST")
        & (transport_master["Transport_Mode"] == "Standard")
    )
    |
    (
        (transport_master["Destination_DC_ID"] == "DC_WEST")
        & (transport_master["Transport_Mode"] == "Expedited")
    )
].copy()

assert len(selected_lanes) == 2

# Starting DC inventory covers Week 1.
# Shipments will replenish demand in Weeks 2-12.
shipments = regional.loc[
    regional["Week"].between(2, 12),
    ["Week", "DC_ID", "SKU", "Demand_Units"]
].copy()

shipments = shipments.rename(columns={
    "Week": "Arrival_Week",
    "Demand_Units": "Ship_Units"
})

# Add transportation parameters
shipments = shipments.merge(
    selected_lanes[[
        "Lane_ID",
        "Destination_DC_ID",
        "Transport_Mode",
        "Cost_Per_Unit_USD",
        "Transit_Time_Weeks",
        "Weekly_Lane_Capacity_Units"
    ]],
    left_on="DC_ID",
    right_on="Destination_DC_ID",
    how="left",
    validate="many_to_one"
)

# Calculate when the shipment must depart
shipments["Ship_Week"] = (
    shipments["Arrival_Week"]
    - shipments["Transit_Time_Weeks"]
)

# Calculate transportation cost
shipments["Freight_Cost_USD"] = (
    shipments["Ship_Units"]
    * shipments["Cost_Per_Unit_USD"]
)

shipments = shipments.sort_values(
    ["Ship_Week", "DC_ID"]
).reset_index(drop=True)

# Validate transportation schedule
assert shipments.notna().all().all()
assert len(shipments) == 22

assert shipments["Ship_Week"].between(1, 12).all()

assert (
    shipments["Ship_Units"]
    <= shipments["Weekly_Lane_Capacity_Units"]
).all()

# Verify plant production covers weekly departures
weekly_shipments = shipments.groupby(
    "Ship_Week"
)["Ship_Units"].sum()

weekly_shipments = weekly_shipments.reindex(
    range(1, 13), fill_value=0
)

weekly_production = production_plan.set_index(
    "Week"
)["Production_Units"].reindex(range(1, 13))

assert np.allclose(
    weekly_shipments.to_numpy(),
    weekly_production.to_numpy()
)

print("Baseline transportation schedule created!")
print("Shipment records:", len(shipments))
print("Total shipped units:", shipments["Ship_Units"].sum())
print("Total freight cost: $",
      round(shipments["Freight_Cost_USD"].sum(), 2))

display(shipments.head(10))


Baseline transportation schedule created!
Shipment records: 22
Total shipped units: 5665
Total freight cost: $ 34556.5


,Arrival_Week,DC_ID,SKU,Ship_Units,Lane_ID,Destination_DC_ID,Transport_Mode,Cost_Per_Unit_USD,Transit_Time_Weeks,Weekly_Lane_Capacity_Units,Ship_Week,Freight_Cost_USD
0,2,DC_EAST,AIRPODS_REFERENCE,309,LANE_001,DC_EAST,Standard,3.5,1,350,1,1081.5
1,2,DC_WEST,AIRPODS_REFERENCE,206,LANE_004,DC_WEST,Expedited,10.0,1,250,1,2060.0
2,3,DC_EAST,AIRPODS_REFERENCE,309,LANE_001,DC_EAST,Standard,3.5,1,350,2,1081.5
3,3,DC_WEST,AIRPODS_REFERENCE,206,LANE_004,DC_WEST,Expedited,10.0,1,250,2,2060.0
4,4,DC_EAST,AIRPODS_REFERENCE,309,LANE_001,DC_EAST,Standard,3.5,1,350,3,1081.5
5,4,DC_WEST,AIRPODS_REFERENCE,206,LANE_004,DC_WEST,Expedited,10.0,1,250,3,2060.0
6,5,DC_EAST,AIRPODS_REFERENCE,309,LANE_001,DC_EAST,Standard,3.5,1,350,4,1081.5
7,5,DC_WEST,AIRPODS_REFERENCE,206,LANE_004,DC_WEST,Expedited,10.0,1,250,4,2060.0
8,6,DC_EAST,AIRPODS_REFERENCE,309,LANE_001,DC_EAST,Standard,3.5,1,350,5,1081.5
9,6,DC_WEST,AIRPODS_REFERENCE,206,LANE_004,DC_WEST,Expedited,10.0,1,250,5,2060.0


In [6]:
# STEP 6A: Simulate weekly component inventory

bom_master = inputs["bom"]
inventory_master = inputs["inventory"]

# Starting component inventory
starting_components = inventory_master.loc[
    inventory_master["Item_Type"] == "Component"
].set_index("Item_ID")[
    "Beginning_Inventory_Units"
].to_dict()

# Component requirements per finished product
bom_requirements = bom_master.set_index(
    "Component_ID"
)["Units_Per_Finished_Unit"].to_dict()

# Initialize inventory balances
component_stock = starting_components.copy()

component_records = []

for week in range(1, 13):

    production = int(
        production_plan.loc[
            production_plan["Week"] == week,
            "Production_Units"
        ].iloc[0]
    )

    for component_id, units_per_product in bom_requirements.items():

        beginning = component_stock[component_id]

        # Supplier deliveries arriving this week
        receipts = purchase_orders.loc[
            (purchase_orders["Arrival_Week"] == week)
            & (purchase_orders["Component_ID"] == component_id),
            "Order_Units"
        ].sum()

        # Components consumed in production
        consumption = production * units_per_product

        available = beginning + receipts

        # Prevent using components before arrival
        assert available >= consumption, (
            f"Component shortage: {component_id}, Week {week}"
        )

        ending = available - consumption

        assert ending >= 0

        component_records.append({
            "Week": week,
            "Component_ID": component_id,
            "Beginning_Inventory": beginning,
            "Supplier_Receipts": receipts,
            "Production_Consumption": consumption,
            "Ending_Inventory": ending
        })

        component_stock[component_id] = ending

component_ledger = pd.DataFrame(component_records)

print("Component inventory simulation completed!")
print("Records:", len(component_ledger))

print("\nFINAL COMPONENT INVENTORY:")
display(
    component_ledger.loc[
        component_ledger["Week"] == 12
    ]
)

print("\nFIRST 12 INVENTORY RECORDS:")
display(component_ledger.head(12))


Component inventory simulation completed!
Records: 36

FINAL COMPONENT INVENTORY:


,Week,Component_ID,Beginning_Inventory,Supplier_Receipts,Production_Consumption,Ending_Inventory
33,12,ELEC_001,0,0,0,0
34,12,BATT_001,0,0,0,0
35,12,ENCL_001,0,0,0,0



FIRST 12 INVENTORY RECORDS:


,Week,Component_ID,Beginning_Inventory,Supplier_Receipts,Production_Consumption,Ending_Inventory
0,1,ELEC_001,1100,0,515,585
1,1,BATT_001,1100,0,515,585
2,1,ENCL_001,1100,0,515,585
3,2,ELEC_001,585,65,515,135
4,2,BATT_001,585,65,515,135
5,2,ENCL_001,585,65,515,135
6,3,ELEC_001,135,515,515,135
7,3,BATT_001,135,515,515,135
8,3,ENCL_001,135,515,515,135
9,4,ELEC_001,135,515,515,135


In [7]:
# STEP 6B: Simulate plant and DC finished goods inventory

# Beginning finished goods at plant
plant_inventory = int(
    inventory_master.loc[
        (inventory_master["Location_ID"] == "PLANT_01")
        & (inventory_master["Item_Type"] == "Finished_Good"),
        "Beginning_Inventory_Units"
    ].iloc[0]
)

# Starting inventory at each DC
dc_stock = inventory_master.loc[
    (inventory_master["Item_Type"] == "Finished_Good")
    & (inventory_master["Location_ID"] != "PLANT_01")
].set_index("Location_ID")[
    "Beginning_Inventory_Units"
].to_dict()

dc_master = inputs["dcs"].set_index("DC_ID")

plant_records = []
dc_records = []

for week in range(1, 13):

    # Manufacturing finished goods flow
    production = int(
        production_plan.loc[
            production_plan["Week"] == week,
            "Production_Units"
        ].iloc[0]
    )

    shipped = shipments.loc[
        shipments["Ship_Week"] == week,
        "Ship_Units"
    ].sum()

    plant_beginning = plant_inventory
    plant_available = plant_beginning + production

    assert shipped <= plant_available, (
        f"Insufficient plant finished goods in Week {week}"
    )

    plant_inventory = plant_available - shipped

    plant_records.append({
        "Week": week,
        "Beginning_Inventory": plant_beginning,
        "Production_Units": production,
        "Outbound_Shipments": shipped,
        "Ending_Inventory": plant_inventory
    })

    # Regional distribution inventory flow
    for dc_id in dc_stock:

        beginning = dc_stock[dc_id]

        arrivals = shipments.loc[
            (shipments["Arrival_Week"] == week)
            & (shipments["DC_ID"] == dc_id),
            "Ship_Units"
        ].sum()

        customer_demand = regional.loc[
            (regional["Week"] == week)
            & (regional["DC_ID"] == dc_id),
            "Demand_Units"
        ].sum()

        available = beginning + arrivals

        capacity = dc_master.loc[
            dc_id, "Storage_Capacity_Units"
        ]

        # Capacity is checked after receipts,
        # before customer demand is fulfilled.
        assert available <= capacity, (
            f"DC storage capacity exceeded: {dc_id}, Week {week}"
        )

        fulfilled = min(available, customer_demand)
        shortage = customer_demand - fulfilled
        ending = available - fulfilled

        assert ending >= 0

        dc_records.append({
            "Week": week,
            "DC_ID": dc_id,
            "Beginning_Inventory": beginning,
            "Shipment_Arrivals": arrivals,
            "Customer_Demand": customer_demand,
            "Fulfilled_Units": fulfilled,
            "Shortage_Units": shortage,
            "Ending_Inventory": ending
        })

        dc_stock[dc_id] = ending

plant_ledger = pd.DataFrame(plant_records)
dc_ledger = pd.DataFrame(dc_records)

print("Finished goods simulation completed!")

print("\nPLANT INVENTORY:")
display(plant_ledger)

print("\nDISTRIBUTION INVENTORY:")
display(dc_ledger.head(12))

print("\nTOTAL CUSTOMER DEMAND:",
      dc_ledger["Customer_Demand"].sum())

print("TOTAL FULFILLED:",
      dc_ledger["Fulfilled_Units"].sum())

print("TOTAL SHORTAGES:",
      dc_ledger["Shortage_Units"].sum())

print("FINAL DC INVENTORY:")
display(dc_ledger.loc[dc_ledger["Week"] == 12])


Finished goods simulation completed!

PLANT INVENTORY:


,Week,Beginning_Inventory,Production_Units,Outbound_Shipments,Ending_Inventory
0,1,0,515,515,0
1,2,0,515,515,0
2,3,0,515,515,0
3,4,0,515,515,0
4,5,0,515,515,0
5,6,0,515,515,0
6,7,0,515,515,0
7,8,0,515,515,0
8,9,0,515,515,0
9,10,0,515,515,0



DISTRIBUTION INVENTORY:


,Week,DC_ID,Beginning_Inventory,Shipment_Arrivals,Customer_Demand,Fulfilled_Units,Shortage_Units,Ending_Inventory
0,1,DC_EAST,330,0,309,309,0,21
1,1,DC_WEST,230,0,206,206,0,24
2,2,DC_EAST,21,309,309,309,0,21
3,2,DC_WEST,24,206,206,206,0,24
4,3,DC_EAST,21,309,309,309,0,21
5,3,DC_WEST,24,206,206,206,0,24
6,4,DC_EAST,21,309,309,309,0,21
7,4,DC_WEST,24,206,206,206,0,24
8,5,DC_EAST,21,309,309,309,0,21
9,5,DC_WEST,24,206,206,206,0,24



TOTAL CUSTOMER DEMAND: 6180
TOTAL FULFILLED: 6180
TOTAL SHORTAGES: 0
FINAL DC INVENTORY:


,Week,DC_ID,Beginning_Inventory,Shipment_Arrivals,Customer_Demand,Fulfilled_Units,Shortage_Units,Ending_Inventory
22,12,DC_EAST,21,309,309,309,0,21
23,12,DC_WEST,24,206,206,206,0,24


In [8]:
# STEP 6C: Validate baseline inventory flows

# Confirm full customer fulfillment
assert dc_ledger["Shortage_Units"].sum() == 0

assert np.isclose(
    dc_ledger["Fulfilled_Units"].sum(),
    inputs["regional"]["Demand_Units"].sum()
)

# No negative inventories
assert component_ledger["Ending_Inventory"].ge(0).all()
assert plant_ledger["Ending_Inventory"].ge(0).all()
assert dc_ledger["Ending_Inventory"].ge(0).all()

# Verify component inventory conservation
for component_id, group in component_ledger.groupby(
    "Component_ID"
):
    starting = starting_components[component_id]
    receipts = group["Supplier_Receipts"].sum()
    consumption = group["Production_Consumption"].sum()
    ending = group.iloc[-1]["Ending_Inventory"]

    assert np.isclose(
        starting + receipts - consumption,
        ending
    )

# Verify finished goods conservation
beginning_finished_goods = (
    inventory_master.loc[
        inventory_master["Item_Type"] == "Finished_Good",
        "Beginning_Inventory_Units"
    ].sum()
)

total_production = production_plan["Production_Units"].sum()
total_fulfilled = dc_ledger["Fulfilled_Units"].sum()

ending_finished_goods = (
    plant_ledger.iloc[-1]["Ending_Inventory"]
    + dc_ledger.loc[
        dc_ledger["Week"] == 12,
        "Ending_Inventory"
    ].sum()
)

assert np.isclose(
    beginning_finished_goods + total_production,
    total_fulfilled + ending_finished_goods
)

print("BASELINE NETWORK VALIDATION")
print("=" * 45)
print("Beginning finished goods:", beginning_finished_goods)
print("Production:", total_production)
print("Customer fulfillment:", total_fulfilled)
print("Ending finished goods:", ending_finished_goods)
print("All inventory balances: PASSED")
print("Lead-time availability checks: PASSED")
print("Capacity checks: PASSED")
print("Baseline simulation: PASSED")

BASELINE NETWORK VALIDATION
Beginning finished goods: 560
Production: 5665
Customer fulfillment: 6180
Ending finished goods: 45
All inventory balances: PASSED
Lead-time availability checks: PASSED
Capacity checks: PASSED
Baseline simulation: PASSED


In [9]:
# STEP 7A: Calculate baseline supply network costs

import pandas as pd
import numpy as np

# 1. Procurement cost by order week
procurement_weekly = (
    purchase_orders.groupby("Week", as_index=False)
    ["Purchase_Cost_USD"].sum()
    .rename(columns={
        "Purchase_Cost_USD": "Procurement_Cost_USD"
    })
)

# 2. Manufacturing cost by week
manufacturing_weekly = production_plan.merge(
    inputs["plants"][[
        "Plant_ID",
        "Assembly_Cost_Per_Unit_USD"
    ]],
    on="Plant_ID",
    how="left",
    validate="many_to_one"
)

assert manufacturing_weekly[
    "Assembly_Cost_Per_Unit_USD"
].notna().all()

manufacturing_weekly["Manufacturing_Cost_USD"] = (
    manufacturing_weekly["Production_Units"]
    * manufacturing_weekly["Assembly_Cost_Per_Unit_USD"]
)

manufacturing_weekly = manufacturing_weekly[[
    "Week", "Manufacturing_Cost_USD"
]]

# 3. Transportation cost by shipment departure week
freight_weekly = (
    shipments.groupby("Ship_Week", as_index=False)
    ["Freight_Cost_USD"].sum()
    .rename(columns={"Ship_Week": "Week"})
)

# 4. Distribution center inventory holding cost
dc_costs = dc_ledger.merge(
    inputs["dcs"][[
        "DC_ID",
        "Holding_Cost_Per_Unit_Per_Week_USD"
    ]],
    on="DC_ID",
    how="left",
    validate="many_to_one"
)

assert dc_costs[
    "Holding_Cost_Per_Unit_Per_Week_USD"
].notna().all()

dc_costs["Holding_Cost_USD"] = (
    dc_costs["Ending_Inventory"]
    * dc_costs["Holding_Cost_Per_Unit_Per_Week_USD"]
)

holding_weekly = (
    dc_costs.groupby("Week", as_index=False)
    ["Holding_Cost_USD"].sum()
)

# 5. Combine all costs into one weekly table
weekly_costs = pd.DataFrame({
    "Week": range(1, 13)
})

for cost_table in [
    procurement_weekly,
    manufacturing_weekly,
    freight_weekly,
    holding_weekly
]:
    weekly_costs = weekly_costs.merge(
        cost_table,
        on="Week",
        how="left",
        validate="one_to_one"
    )

cost_columns = [
    "Procurement_Cost_USD",
    "Manufacturing_Cost_USD",
    "Freight_Cost_USD",
    "Holding_Cost_USD"
]

weekly_costs[cost_columns] = (
    weekly_costs[cost_columns].fillna(0)
)

weekly_costs["Total_Cost_USD"] = (
    weekly_costs[cost_columns].sum(axis=1)
)

print("BASELINE COST CALCULATION COMPLETE")
print("=" * 45)

display(weekly_costs)

print("\nTOTAL COST BY CATEGORY:")
display(weekly_costs[cost_columns].sum())

print(
    "\nTOTAL BASELINE COST: $",
    round(weekly_costs["Total_Cost_USD"].sum(), 2)
)


BASELINE COST CALCULATION COMPLETE


,Week,Procurement_Cost_USD,Manufacturing_Cost_USD,Freight_Cost_USD,Holding_Cost_USD,Total_Cost_USD
0,1,28785.0,11330,3141.5,59.85,43316.35
1,2,28785.0,11330,3141.5,59.85,43316.35
2,3,28785.0,11330,3141.5,59.85,43316.35
3,4,28785.0,11330,3141.5,59.85,43316.35
4,5,28785.0,11330,3141.5,59.85,43316.35
5,6,28785.0,11330,3141.5,59.85,43316.35
6,7,28785.0,11330,3141.5,59.85,43316.35
7,8,28440.0,11330,3141.5,59.85,42971.35
8,9,24300.0,11330,3141.5,59.85,38831.35
9,10,0.0,11330,3141.5,59.85,14531.35



TOTAL COST BY CATEGORY:


Procurement_Cost_USD      254235.0
Manufacturing_Cost_USD    124630.0
Freight_Cost_USD           34556.5
Holding_Cost_USD             718.2
dtype: float64


TOTAL BASELINE COST: $ 414139.7


In [10]:
# STEP 7B: Calculate baseline operating KPIs

# Demand and fulfillment
total_demand = dc_ledger["Customer_Demand"].sum()
total_fulfilled = dc_ledger["Fulfilled_Units"].sum()
total_shortages = dc_ledger["Shortage_Units"].sum()

service_level = (
    total_fulfilled / total_demand * 100
)

# Production and plant utilization
total_production = production_plan[
    "Production_Units"
].sum()

plant_capacity_12wk = (
    inputs["plants"]["Weekly_Capacity_Units"].sum()
    * len(production_plan)
)

plant_utilization = (
    total_production / plant_capacity_12wk * 100
)

# Supplier utilization across the full 12 weeks
supplier_orders = (
    purchase_orders.groupby("Supplier_ID")
    ["Order_Units"].sum()
)

supplier_capacity = (
    inputs["suppliers"].groupby("Supplier_ID")
    ["Weekly_Capacity_Units"].sum()
    * len(production_plan)
)

supplier_utilization = pd.DataFrame({
    "Ordered_Units": supplier_orders,
    "Available_Capacity_Units": supplier_capacity
})

supplier_utilization["Utilization_Pct"] = (
    supplier_utilization["Ordered_Units"]
    / supplier_utilization["Available_Capacity_Units"]
    * 100
)

# Transportation mode breakdown
freight_modes = shipments.groupby(
    "Transport_Mode"
)["Ship_Units"].sum()

expedited_units = freight_modes.get(
    "Expedited", 0
)

expedited_share = (
    expedited_units / shipments["Ship_Units"].sum()
    * 100
)

# Cost metrics
total_cost = weekly_costs["Total_Cost_USD"].sum()
cost_per_fulfilled_unit = total_cost / total_fulfilled

# Inventory metrics
average_dc_inventory = (
    dc_ledger.groupby("Week")
    ["Ending_Inventory"].sum().mean()
)

ending_dc_inventory = dc_ledger.loc[
    dc_ledger["Week"] == 12,
    "Ending_Inventory"
].sum()

# Executive KPI table
baseline_kpis = pd.DataFrame([
    ["Total Demand", total_demand, "Units"],
    ["Fulfilled Demand", total_fulfilled, "Units"],
    ["Shortages", total_shortages, "Units"],
    ["Service Level", service_level, "%"],
    ["Total Production", total_production, "Units"],
    ["Plant Utilization", plant_utilization, "%"],
    ["Expedited Freight Share", expedited_share, "%"],
    ["Average Ending DC Inventory",
     average_dc_inventory, "Units"],
    ["Final DC Inventory", ending_dc_inventory, "Units"],
    ["Total Modeled Cost", total_cost, "USD"],
    ["Modeled Cost Per Fulfilled Unit",
     cost_per_fulfilled_unit, "USD/Unit"]
], columns=[
    "Metric", "Value", "Unit"
])

print("BASELINE OPERATING KPIs")
display(baseline_kpis.round(2))

print("\nSUPPLIER UTILIZATION")
display(supplier_utilization.round(2))


BASELINE OPERATING KPIs


,Metric,Value,Unit
0,Total Demand,6180.00,Units
1,Fulfilled Demand,6180.00,Units
2,Shortages,0.00,Units
3,Service Level,100.00,%
4,Total Production,5665.00,Units
5,Plant Utilization,81.39,%
6,Expedited Freight Share,40.00,%
7,Average Ending DC Inventory,45.00,Units
8,Final DC Inventory,45.00,Units
9,Total Modeled Cost,414139.70,USD



SUPPLIER UTILIZATION


,Ordered_Units,Available_Capacity_Units,Utilization_Pct
Supplier_ID,,,
SUP_A,12150,16200,75.00
SUP_B,1545,9000,17.17


In [11]:
# STEP 7C: Validate baseline financial and operating KPIs

# Validate cost category totals
assert np.isclose(
    weekly_costs["Procurement_Cost_USD"].sum(),
    purchase_orders["Purchase_Cost_USD"].sum()
)

assert np.isclose(
    weekly_costs["Manufacturing_Cost_USD"].sum(),
    total_production * 22
)

assert np.isclose(
    weekly_costs["Freight_Cost_USD"].sum(),
    shipments["Freight_Cost_USD"].sum()
)

assert np.isclose(
    weekly_costs["Holding_Cost_USD"].sum(),
    dc_costs["Holding_Cost_USD"].sum()
)

# No negative costs
assert (
    weekly_costs[[
        "Procurement_Cost_USD",
        "Manufacturing_Cost_USD",
        "Freight_Cost_USD",
        "Holding_Cost_USD"
    ]] >= 0
).all().all()

# Financial reconciliation
assert np.isclose(
    total_cost,
    weekly_costs[[
        "Procurement_Cost_USD",
        "Manufacturing_Cost_USD",
        "Freight_Cost_USD",
        "Holding_Cost_USD"
    ]].sum().sum()
)

# Customer service reconciliation
assert np.isclose(
    total_demand,
    total_fulfilled + total_shortages
)

assert 0 <= service_level <= 100
assert 0 <= plant_utilization <= 100

assert supplier_utilization[
    "Utilization_Pct"
].between(0, 100).all()

# Validate record structure
assert len(weekly_costs) == 12
assert weekly_costs["Week"].is_unique
assert baseline_kpis["Metric"].is_unique
assert baseline_kpis["Value"].notna().all()

print("BASELINE COST AND KPI VALIDATION")
print("=" * 45)
print("Procurement reconciliation: PASSED")
print("Manufacturing reconciliation: PASSED")
print("Freight reconciliation: PASSED")
print("Holding cost reconciliation: PASSED")
print("Customer service reconciliation: PASSED")
print("Supplier utilization checks: PASSED")
print("All validation checks: PASSED")


BASELINE COST AND KPI VALIDATION
Procurement reconciliation: PASSED
Manufacturing reconciliation: PASSED
Freight reconciliation: PASSED
Holding cost reconciliation: PASSED
Customer service reconciliation: PASSED
Supplier utilization checks: PASSED
All validation checks: PASSED


In [12]:
# STEP 7D: Export baseline planning results

from pathlib import Path

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

baseline_outputs = {
    "baseline_purchase_orders.csv": purchase_orders,
    "baseline_production_plan.csv": production_plan,
    "baseline_shipments.csv": shipments,
    "baseline_component_inventory.csv": component_ledger,
    "baseline_plant_inventory.csv": plant_ledger,
    "baseline_dc_inventory.csv": dc_ledger,
    "baseline_weekly_costs.csv": weekly_costs,
    "baseline_kpis.csv": baseline_kpis,
    "baseline_supplier_utilization.csv":
        supplier_utilization.reset_index()
}

for filename, dataframe in baseline_outputs.items():

    output_file = output_dir / filename

    dataframe.to_csv(output_file, index=False)

    assert output_file.is_file()

    print(f"Exported: {filename}")

print("\nSUCCESS: All baseline outputs exported!")
print("Files created:", len(baseline_outputs))
print("Output directory:", output_dir)


Exported: baseline_purchase_orders.csv
Exported: baseline_production_plan.csv
Exported: baseline_shipments.csv
Exported: baseline_component_inventory.csv
Exported: baseline_plant_inventory.csv
Exported: baseline_dc_inventory.csv
Exported: baseline_weekly_costs.csv
Exported: baseline_kpis.csv
Exported: baseline_supplier_utilization.csv

SUCCESS: All baseline outputs exported!
Files created: 9
Output directory: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tables


In [13]:
# STEP 8: Final baseline validation and summary

import numpy as np
import pandas as pd

# Financial reconciliation
procurement = weekly_costs["Procurement_Cost_USD"].sum()
manufacturing = weekly_costs["Manufacturing_Cost_USD"].sum()
transportation = weekly_costs["Freight_Cost_USD"].sum()
holding = weekly_costs["Holding_Cost_USD"].sum()

calculated_total = (
    procurement
    + manufacturing
    + transportation
    + holding
)

assert np.isclose(calculated_total, total_cost)

# Service and inventory validation
assert total_demand == 6180
assert total_fulfilled == 6180
assert total_shortages == 0

assert np.isclose(
    beginning_finished_goods + total_production,
    total_fulfilled + ending_finished_goods
)

# Verify output files exist
for filename in baseline_outputs:
    assert (output_dir / filename).is_file(), filename

# Build final summary
final_summary = pd.DataFrame({
    "Metric": [
        "Total Demand",
        "Fulfilled Demand",
        "Service Level (%)",
        "Total Production",
        "Beginning Finished Goods",
        "Ending Finished Goods",
        "Procurement Cost ($)",
        "Manufacturing Cost ($)",
        "Transportation Cost ($)",
        "DC Holding Cost ($)",
        "Total Modeled Cost ($)",
        "Cost Per Fulfilled Unit ($)"
    ],
    "Value": [
        total_demand,
        total_fulfilled,
        service_level,
        total_production,
        beginning_finished_goods,
        ending_finished_goods,
        procurement,
        manufacturing,
        transportation,
        holding,
        calculated_total,
        calculated_total / total_fulfilled
    ]
})

display(final_summary.round(2))

print("\nNOTEBOOK 03 FINAL VALIDATION: PASSED")
print("Baseline ready for optimization comparison.")


,Metric,Value
0,Total Demand,6180.00
1,Fulfilled Demand,6180.00
2,Service Level (%),100.00
3,Total Production,5665.00
4,Beginning Finished Goods,560.00
5,Ending Finished Goods,45.00
6,Procurement Cost ($),254235.00
7,Manufacturing Cost ($),124630.00
8,Transportation Cost ($),34556.50
9,DC Holding Cost ($),718.20



NOTEBOOK 03 FINAL VALIDATION: PASSED
Baseline ready for optimization comparison.



## Final Summary — Baseline Supply Plan

### Business Objective
Evaluate the operating cost and customer service
performance of a rule-based hardware supply plan.

### Methodology
A 12-week supply chain simulation was developed
using predefined procurement, production, and
transportation policies.

The simulation incorporated:
- Supplier purchasing lead times
- Supplier component capacity
- Manufacturing capacity and BOM consumption
- Transportation modes and transit times
- Distribution center inventory balances
- Regional customer demand

### Baseline Results
Under the hypothetical V1 assumptions:

- Total customer demand: 6,180 units
- Total production: 5,665 units
- Customer fulfillment: 100%
- Ending finished goods inventory: 45 units
- Total modeled operating cost: $414,139.70

### Key Findings
1. The baseline meets all modeled customer demand.
2. Starting inventory supports early-week fulfillment.
3. Supplier lead times influence procurement timing.
4. Expedited transportation increases cost but
   supports the chosen replenishment schedule.
5. The baseline provides a feasible benchmark
   for mathematical optimization.

### Limitations
The model uses historical illustrative demand
and hypothetical supply network assumptions.

Modeled operating costs exclude acquisition
costs of initial inventory and unspecified
component or plant inventory holding costs.

The baseline is not necessarily cost-optimal.

### Next Steps
Develop a mathematical optimization model using
Pyomo and HiGHS to minimize supply network costs
while maintaining operational feasibility.

Compare the optimized and baseline plans using
consistent costs, inventory requirements,
capacities, lead times, and customer service.
